# Reinforcement‑Learning Argument Mining

In [ ]:
!hf auth login

In [ ]:
import os
import re
import json
import difflib

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
CHECKPOINT_ID = "oberbics/llama-8B-argument_mining_V6"
BASE_MODEL_ID = "oberbics/llama-3.1-newspaper-arguments-your_name-optimized_full"
DATA_FILE = "post-training_multiarg_final.txt"

TARGET_STEPS = 600
STEPS_PER_JOB = 4
NUM_GENERATIONS = 4
NUM_ARTICLES = 150          # same 150-article pool as in V5

SYSTEM_PROMPT = '''You are an expert at analyzing historical texts and you hate to summarize

CRITICAL QUALITY REQUIREMENTS:
- Extract EXACT text from source (no paraphrasing)
- Infer PRECISE claims (what the argument really implies)
- Provide CLEAR reasoning (why this is an argument)
- Judge uncertainty CORRECTLY (when human verification needed)
- NEVER REPEAT ARGUMENTS - Each argument must be COMPLETELY UNIQUE

CRITICAL: Many texts contain NO arguments - just factual reporting!
- Historical facts like "The earthquake occurred at 5:20 AM" are NOT arguments
- Casualty reports like "500 people died" are NOT arguments
- Weather descriptions like "Heavy rain fell" are NOT arguments
- Event descriptions without persuasive intent are NOT arguments
- When in doubt, output NA for all fields!
- FALSE POSITIVES (claiming arguments where none exist) are HEAVILY penalized!
- If you see only factual reporting with no persuasive elements, return NA!

The <human_verification_needed> field is EXTREMELY IMPORTANT!
- Set to TRUE when: argument is ambiguous, implicit, or you're uncertain
- Set to FALSE when: argument is explicit, clear, and you're confident
- WRONG verification settings will SEVERELY reduce your score!

OUTPUT FORMAT - EXACTLY these 4 XML tags and NOTHING else:
<argument>Original argument text OR "NA"</argument>
<claim>Core claim (implication) in one sentence OR "NA"</claim>
<explanation>Why this is an argument OR "NA"</explanation>
<human_verification_needed>True OR False</human_verification_needed>

EXAMPLE WITH ARGUMENT:
<argument>It is reported from Malta that the British war-ships "Exmouth," "Euryalus," "Minerva," and "Sutlej" have left for Messina. The French Government has sent two armoured ships and three destroyers to Messina. President Fallieres, Premier Clemenceau, Minister Pichon, and the Presidents of the Senate and Chamber have all sent messages of sympathy to the Italian Government. The help already proffered and accepted is insufficient for the purpose. There is pressing need of extraordinary measures of help, and provisions are in great demand. There is need of doctors, tents, clothing, and provisions for the survivors, who, deprived of all necessities, are exposed to the inclemencies of the winter weather. There is need of fire engines to cope with the flames that are raging among the ruins. The railway station has collapsed. Railway carriages have been destroyed. Almost all the railway employees are dead. The streets are no longer recognisable; they look like enormous fissures in a distant and extensive heap of ruins.</argument>
<claim>Current relief efforts are inadequate and much more extensive aid is urgently needed.</claim>
<explanation>The prefect explicitly argues that existing help is "insufficient" and makes a direct claim that "extraordinary measures" are needed, presenting a clear premise-conclusion structure about the inadequacy of current response.</explanation>
<human_verification_needed>False</human_verification_needed>

EXAMPLE WITHOUT ARGUMENT (just factual reporting):
<argument>NA</argument>
<claim>NA</claim>
<explanation>NA</explanation>
<human_verification_needed>False</human_verification_needed>

RULES:
- NEVER REPEAT ARGUMENTS - Each argument must be COMPLETELY UNIQUE
- TRUE FOR VERIFICATION WHEN UNCERTAIN
- Only output arguments that appear verbatim (or nearly verbatim) in the text
- NO SUMMARY; ONLY EXACT EXTRACTION FROM THE TEXT; don't extract anything that is not in the text. Only extract word by word
- ONLY output these 4 XML tags
- Extract only original text without changes or use NA when you did not find an argument
- Factual reportings such as "Dem Vulkanausbruch folgten drei Sturzwellen in etwa 10 Meter Höhe" or "Almost all the inhabitants were killed; only a few thousands escaped death" are NO Arguments
- The CLAIM should say what the (implicite) argument implies, what the main conclusion is
- Give attention to implicit arguments
- In cases of uncertainty or ambiguity, say human_verification_needed TRUE
- If no argument exists, use NA for ALL fields without explanation except <human_verification_needed>FALSE or TRUE</human_verification_needed>
- More than one argument possible for one article, one unit has one clear claim and all the xml structures
- Better to miss a weak argument than to create a false positive!

VERIFICATION: Before you print the results, double check claims and explanations of the argument. When the claim is just a translation or the explanation states that this is not an argument, don't print it. ALSO verify this argument is NOT a duplicate of any previous argument in your output.'''

USER_PREFIX = "Extract argumentative units from historical text in their original form, no summaries.\n"

# ---------------------------------------------------------------------------
# Parsing and similarity
# ---------------------------------------------------------------------------
PATTERN = re.compile(
    r"<argument>(.*?)</argument>\s*<claim>(.*?)</claim>\s*"
    r"<explanation>(.*?)</explanation>\s*"
    r"<human_verification_needed>(.*?)</human_verification_needed>",
    re.DOTALL | re.IGNORECASE,
)


def clean(text):
    return " ".join(str(text).split())


def norm(text):
    return clean(text).lower()


def sim(a, b):
    return difflib.SequenceMatcher(None, norm(a), norm(b), autojunk=False).ratio()


def extract_individual_arguments(xml_text):
    arguments = []
    for arg, claim, expl, verif in PATTERN.findall(str(xml_text)):
        arg_text = clean(arg)
        if arg_text.upper() == "NA":
            continue
        arguments.append({
            "argument": arg_text,
            "claim": clean(claim),
            "explanation": clean(expl),
            "verification": verif.strip().lower(),
        })
    return arguments


def detect_and_remove_duplicates(model_args):
    unique_args, duplicate_count = [], 0
    for arg in model_args:
        if any(sim(arg["argument"], u["argument"]) > 0.80 for u in unique_args):
            duplicate_count += 1
        else:
            unique_args.append(arg)
    return unique_args, duplicate_count


def component_similarity(model_arg, gt_arg):
    verif_match = 1.0 if model_arg["verification"] == gt_arg["verification"] else 0.0
    return (sim(model_arg["argument"], gt_arg["argument"]) * 0.40
            + sim(model_arg["claim"], gt_arg["claim"]) * 0.30
            + sim(model_arg["explanation"], gt_arg["explanation"]) * 0.15
            + verif_match * 0.15)


# ---------------------------------------------------------------------------
# Reward
# ---------------------------------------------------------------------------
def score_one(content, gt):
    """Reward for one model answer against the annotation of ITS OWN article."""
    model_args, duplicate_count = detect_and_remove_duplicates(extract_individual_arguments(content))
    gt_args = extract_individual_arguments(gt)
    num_model, num_gt = len(model_args), len(gt_args)
    duplicate_penalty = duplicate_count * -0.6

    if num_model == 0 and num_gt == 0:
        reward = 0.1
    elif num_model == 0:
        reward = -0.15
    elif num_gt == 0:
        reward = -0.15 * num_model
        reward += -0.15 * sum(1 for a in model_args if a["verification"] == "false")
    else:
        match_scores, used = [], set()
        for gt_arg in gt_args:
            best_score, best_idx = 0.0, None
            for idx, model_arg in enumerate(model_args):
                if idx in used:
                    continue
                score = component_similarity(model_arg, gt_arg)
                if score > best_score:
                    best_score, best_idx = score, idx
            if best_idx is not None and best_score >= 0.25:
                match_scores.append(best_score)
                used.add(best_idx)
        missed = num_gt - len(match_scores)
        extra = num_model - len(used)
        reward = (sum(match_scores) - 0.15 * missed - 0.10 * extra) / num_gt

    return max(-3.0, min(2.0, reward + duplicate_penalty))


reward_call_count = 0


def ground_truth_reward_func(completions, ground_truth=None, example_id=None, **kwargs):
    """TRL passes one ground_truth / example_id entry PER COMPLETION."""
    global reward_call_count
    reward_call_count += 1
    if ground_truth is None or len(ground_truth) != len(completions):
        raise ValueError("ground_truth must be passed with one entry per completion")

    rewards = []
    for i, comp in enumerate(completions):
        content = comp[0]["content"] if isinstance(comp, (list, tuple)) else str(comp)
        rewards.append(float(score_one(content, ground_truth[i])))

    if reward_call_count <= 5 or reward_call_count % 20 == 0:
        ids = example_id if example_id is not None else ["?"] * len(rewards)
        print(f"[reward call {reward_call_count}] " +
              ", ".join(f"{e}: {r:+.2f}" for e, r in zip(ids, rewards)))
    return rewards


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def load_examples(path):
    """One JSON object per line. Broken lines are skipped and REPORTED, never silently dropped."""
    objs, bad = [], []
    with open(path, encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                objs.append(json.loads(line))
            except json.JSONDecodeError:
                bad.append(n)
    if bad:
        print(f"WARNING: {len(bad)} unreadable line(s) skipped: {bad[:10]}")
    return objs


def reward_self_test(examples, n=20):
    """Abort before any GPU time is spent if the reward is wired up wrongly."""
    checked = 0
    for idx, ex in enumerate(examples[:n]):
        gt = ex["ground_truth"]
        own = score_one(gt, gt)
        expected = 0.1 if not extract_individual_arguments(gt) else 1.0
        assert abs(own - expected) < 1e-6, f"{ex['example_id']}: own annotation scored {own}, expected {expected}"

        other = examples[(idx + 1) % len(examples)]["ground_truth"]
        if extract_individual_arguments(gt) and extract_individual_arguments(other):
            assert score_one(other, gt) < own, f"{ex['example_id']}: wrong article scored as high as the right one"

        labels = {o["quality_label"]: o["output"] for o in ex.get("synthetic_outputs", [])}
        if "PERFECT" in labels and "VERY_POOR" in labels:
            assert score_one(labels["PERFECT"], gt) > score_one(labels["VERY_POOR"], gt), \
                f"{ex['example_id']}: VERY_POOR scored at least as high as PERFECT"
        checked += 1
    print(f"Reward self-test passed on {checked} examples.")


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def main():
    import torch
    from datasets import Dataset
    from huggingface_hub import HfApi, login
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import GRPOConfig, GRPOTrainer

    token = os.environ.get("HF_TOKEN")
    login(token=token)
    api = HfApi()

    steps_done = 0
    try:
        commits = api.list_repo_commits(CHECKPOINT_ID, token=token)
        steps_done = sum("Training in progress" in c.title for c in commits) * STEPS_PER_JOB
    except Exception:
        pass
    print(f"Steps done so far: {steps_done}/{TARGET_STEPS}")
    if steps_done >= TARGET_STEPS:
        print("Target reached.")
        return

    examples = load_examples(DATA_FILE)
    print(f"Loaded {len(examples)} examples from file.")
    examples = [ex for ex in examples[:NUM_ARTICLES] if ex.get("source_text") and ex.get("ground_truth")]
    print(f"Using {len(examples)} articles (first {NUM_ARTICLES}).")
    reward_self_test(examples)

    ds = Dataset.from_dict({
        "example_id": [str(ex.get("example_id", f"example_{i:04d}")) for i, ex in enumerate(examples)],
        "prompt": [[{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": USER_PREFIX + ex["source_text"]}] for ex in examples],
        "ground_truth": [ex["ground_truth"] for ex in examples],
    })

    model_id = CHECKPOINT_ID if steps_done > 0 else BASE_MODEL_ID
    print(f"Loading {model_id}")
    model = AutoModelForCausalLM.from_pretrained(model_id, token=token, torch_dtype=torch.bfloat16, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    args = GRPOConfig(
        output_dir="./grpo-v6",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        max_steps=STEPS_PER_JOB,
        logging_steps=1,
        save_steps=STEPS_PER_JOB,
        save_total_limit=1,
        scale_rewards="group",
        num_generations=NUM_GENERATIONS,
        generation_batch_size=NUM_GENERATIONS,
        temperature=0.8,
        top_p=0.95,
        repetition_penalty=1.05,
        max_completion_length=1200,
        max_prompt_length=5048,
        bf16=True,
        gradient_checkpointing=True,
        remove_unused_columns=False,
        seed=42 + steps_done,
        report_to=[],
        push_to_hub=True,
        hub_model_id=CHECKPOINT_ID,
        hub_token=token,
        optim="paged_adamw_8bit",
    )

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=ground_truth_reward_func,
        args=args,
        train_dataset=ds,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model()
    trainer.push_to_hub()

    with open("log_history.json", "w") as f:
        json.dump(trainer.state.log_history, f, indent=1)
    api.upload_file(
        path_or_fileobj="log_history.json",
        path_in_repo=f"logs/log_history_steps_{steps_done:04d}.json",
        repo_id=CHECKPOINT_ID,
        token=token,
    )
    print(f"Done: {steps_done + STEPS_PER_JOB}/{TARGET_STEPS}")


if __name__ == "__main__":
    main()